<a href="https://colab.research.google.com/github/gavrilovAlikhan/ML-Practice/blob/main/Student%20Scores/student_scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
%pip install catboost lightgbm optuna

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score, cross_val_predict
from sklearn.metrics import root_mean_squared_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.linear_model import LinearRegression, Ridge, SGDRegressor

import optuna

import typing

import warnings

In [ ]:
TARGET = "exam_score"
N_SPLITS = 5
SEED = 42

non_relevant_columns = ['id', 'source']

ordinal_maps = {
    "gender" : {"male":0, "female":1, "other":2},
    "internet_access" : {"no":0, "yes":1},
    "sleep_quality" : {"poor":0, "average":1, "good":2},
    "facility_rating" : {"low":0, "medium":1, "high":2},
    "exam_difficulty" : {"easy":0, "moderate":1, "hard":2},
    "course" : {"ba":0, "b.sc":1, "diploma":2, "b.tech":3, "b.com":4, "bca":5, "bba":6},
    "study_method" : {"self-study":0, "online videos":1, "group study":2, "mixed":3, "coaching":4},
}

original_categorical_cols = ['gender', 'course', 'internet_access', 'sleep_quality', 'study_method', 'facility_rating', 'exam_difficulty']

# Data Loader

In [ ]:
def load_data(option: str="local"):
  '''
  returns train_df, test_df, df

  '''
  if option == "local":
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

  elif option == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    train_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Student Scores/train.csv')
    test_df = pd.read_csv('/content/drive/MyDrive/Kaggle Practice/Student Scores/test.csv')

  train_df['source'] = 'train'
  test_df['source'] = 'test'
  df = pd.concat([train_df, test_df], ignore_index=True)

  return train_df, test_df, df

In [ ]:
df_train, _, df = load_data(option='colab')

# Data Splitter

In [ ]:
def data_spliter(df: pd.DataFrame, additional_drop_columns: list = []) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    '''
    return: X_train, y_train, submission_df

    '''
    train_dataset = df[df['source'] == 'train'].drop(columns=non_relevant_columns + additional_drop_columns)
    submission_dataset = df[df['source'] == 'test'].drop(columns=['source', TARGET] + additional_drop_columns)

    X_train = train_dataset.drop(columns=[TARGET])
    y_train = train_dataset[TARGET]

    return X_train, y_train, submission_dataset

# EDA

In [ ]:
df.isna().sum()

,0
id,0
age,0
gender,0
course,0
study_hours,0
class_attendance,0
internet_access,0
sleep_hours,0
sleep_quality,0
study_method,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900000 entries, 0 to 899999
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                900000 non-null  int64  
 1   age               900000 non-null  int64  
 2   gender            900000 non-null  object 
 3   course            900000 non-null  object 
 4   study_hours       900000 non-null  float64
 5   class_attendance  900000 non-null  float64
 6   internet_access   900000 non-null  object 
 7   sleep_hours       900000 non-null  float64
 8   sleep_quality     900000 non-null  object 
 9   study_method      900000 non-null  object 
 10  facility_rating   900000 non-null  object 
 11  exam_difficulty   900000 non-null  object 
 12  exam_score        630000 non-null  float64
 13  source            900000 non-null  object 
dtypes: float64(4), int64(2), object(8)
memory usage: 96.1+ MB


In [ ]:
df.drop(columns=non_relevant_columns).describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,900000.0,NaN,NaN,NaN,20.545316,2.260301,17.0,19.0,21.0,23.0,24.0
gender,900000,3,male,301275,NaN,NaN,NaN,NaN,NaN,NaN,NaN
course,900000,7,b.tech,187697,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_hours,900000.0,NaN,NaN,NaN,4.0028,2.359238,0.08,1.98,4.0,6.05,7.91
class_attendance,900000.0,NaN,NaN,NaN,71.985836,17.425469,40.6,57.0,72.6,87.2,99.4
internet_access,900000,2,yes,828094,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sleep_hours,900000.0,NaN,NaN,NaN,7.072552,1.745021,4.1,5.6,7.1,8.6,9.9
sleep_quality,900000,3,poor,305355,NaN,NaN,NaN,NaN,NaN,NaN,NaN
study_method,900000,5,coaching,188395,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_rating,900000,3,medium,305898,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
z.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score,source
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3,train
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7,train
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0,train
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9,train
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0,train


## Correlations

In [ ]:
z = df.drop(columns=non_relevant_columns).copy()
z['internet_access'] = z['internet_access'].map({'no':0, 'yes':1})
z['sleep_quality'] = z['sleep_quality'].map({'average':1, 'poor':0, 'good':2})
z['facility_rating'] = z['facility_rating'].map({'medium':1, 'low':0, 'high':2})
z['exam_difficulty'] = z['exam_difficulty'].map({'moderate':1, 'easy':0, 'hard':2})
z = pd.get_dummies(z, columns=['study_method', 'course', 'gender'], dtype=int)

In [ ]:
z_corr = z.corr()
z_sorted_cols = z_corr['exam_score'].abs().sort_values(ascending=False).index.to_list()

z_corr_sorted = z_corr.loc[z_sorted_cols, z_sorted_cols]
mask = np.triu(np.ones_like(z_corr_sorted, dtype=bool), k=1)
z_corr_sorted[mask] = np.nan

### Correlation Plot

In [ ]:
corr_fig = px.imshow(
    z_corr_sorted,
    text_auto = ".2f",
    aspect=True,
    color_continuous_scale="RdBu_r",
    width=1200,
    height=800
)

corr_fig.show()

## Distribution of categorical columns by exam score